In [ ]:
import pandas as pd
import numpy as np

# Configuration pour l'affichage de toutes les colonnes dans le notebook
pd.set_option('display.max_columns', None)

# Lecture et écriture de fichiers CSV avec Pandas

`read_csv` est la fonction la plus utilisée pour lire des fichiers CSV. Elle prend en paramètre le chemin vers le fichier CSV à lire. On peut aussi lui passer un URL. Elle retourne un objet de type `DataFrame` qui est une structure de données de Pandas.

C'est une bonne idée de spécifier l'encodage de caractères du fichier CSV. S'il y a des soucis avec les accents, essayez d'utiliser `latin-1` ou `iso-8859-1`.

Une caractéristique distinctive de pandas par rapport à numpy est l'utilisation d'**index**. Les index permettent d'identifier de manière unique chaque ligne et facilitent les opérations de jointure et de sélection.

In [ ]:

# Lecture du fichier CSV avec définition de l'index
df = pd.read_csv("sample_data.csv", encoding="utf-8", index_col="id")
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

print("DataFrame avec index personnalisé:")
df.head()

# Infos de base

Permet de vérifier rapidement ce à quoi ressemblent les données chargées.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(df.memory_usage(index=True, deep=True))

In [ ]:
print(f"{df.memory_usage(index=True, deep=True).sum() // 1024} kB")

La méthode `drop` permet de supprimer des colonnes ou des lignes d'un DataFrame. On peut spécifier les colonnes à supprimer en utilisant le paramètre `columns` et les lignes à supprimer en utilisant le paramètre `index`. Le paramètre `inplace=True` permet de modifier le DataFrame en place sans créer une copie; sinon, il retourne une nouvelle copie du DataFrame avec les modifications.

Note: plusieurs méthodes de pandas ont un paramètre `inplace` qui permet de modifier le DataFrame en place sans créer une copie. C'est souvent plus efficace en termes de mémoire, mais il faut faire attention car cela modifie le DataFrame original, qui est peut-être utilisé ailleurs dans le code…

In [ ]:
df.drop(columns=['comment'], inplace=True)
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html

La méthode `to_csv` enregistre un fichier: elle prend en paramètre un objet de type `DataFrame` et écrit le contenu dans un fichier CSV. Ici par exemple, on aura le contenu sans la colonne `comment`, qui a été supprimée avec `drop`.

In [ ]:
df.to_csv("output.csv")
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html

# Manipulation des index

Les index pandas sont un élément clé qui distingue pandas de numpy. Ils permettent d'identifier les lignes de manière unique et facilitent les opérations de jointure et de sélection.

In [ ]:
# Exemple de manipulation des index
print("Index actuel:")
print(df.index.name, df.index[:5])

# Réinitialiser l'index (transforme l'index en colonne)
df_reset = df.reset_index()
print("\nAprès reset_index:")
df_reset.head()

In [ ]:
# Définir une nouvelle colonne comme index
df_new_index = df_reset.set_index('computer_science_is')
print("\nAvec un nouvel index (... qui ici n'a pas vraiment de sens):")
df_new_index.head()

# Sélection

On peut utiliser l'accesseur `loc` pour passer l'index (donc une valeur de la colonne déterminée comme étant l'index) d'une ligne. Si on passe un deuxième argument, on peut sélectionner la colonne et ainsi direct accéder à une cellule.

On peut aussi utiliser `iloc` pour accéder à une ligne ou une colonne par son numéro. `iloc` -> integer location.

In [ ]:
print(df.loc[10460])

In [ ]:
print(df.iloc[3])

Si on fait du subscripting directement sur une `DataFrame`, on peut sélectionner une ou plusieurs colonnes, qui sont reetournées sous forme de `Series` ou de `DataFrame`, selon le nombre de colonnes sélectionnées.

In [ ]:
print(df["q1"].sum())

In [ ]:
print(df.loc[10460, ["q1", "q2"]])

Sur des `DataFrame`s et les `Series`, on peut utiliser les opérateurs `+`, `-`, `*`, `/` pour faire des opérations sur toutes les cellules. On peut aussi utiliser les fonctions `add`, `sub`, `mul`, `div` de Pandas.

Des fonctions d'agrégration sont aussi disponibles: `sum`, `mean`, `median`, `min`, `max`, `std`, `var`, `count`, `quantile`, etc. Sur des `DataFrame`s, on peut spécifier avec l'argument `axis` si on veut faire l'aggrégation sur les lignes ou sur les colonnes: `axis=0` pour les colonnes, `axis=1` pour les lignes.

In [ ]:
qs = [f"q{i}" for i in range(1, 33)]
print(qs)
questions = df[qs]
total_by_student = questions.sum(axis=1)
total_by_student

In [ ]:
# Total sur chaque question (moyenne)
total_by_question = questions.sum(axis=0) / questions.shape[0]
total_by_question

# Ajout et suppression de colonnes

On peut modifier une `DataFrame` de plusieurs façons. La méthode `assign()` est préférable à `insert()` car elle permet un style plus fonctionnel et peut être chaînée avec d'autres opérations.

In [ ]:
# Utilisation de assign() pour ajouter une colonne (recommandé)
df_with_total = df.assign(total=total_by_student)
print("DataFrame avec colonne 'total' ajoutée:")
df_with_total.head()

In [ ]:
# Suppression de colonnes avec drop()
qs_to_remove = ["q1", "q2", "q3"]  # Exemple: supprimer les 3 premières questions
df_reduced = df_with_total.drop(columns=qs_to_remove)
print(f"DataFrame après suppression de {qs_to_remove}:")
print(f"Colonnes restantes: {list(df_reduced.columns[:10])}...")  # Afficher les 10 premières

# Alternative avec loc (équivalent mais plus verbeux)
# df_reduced_alt = df_with_total.loc[:, ~df_with_total.columns.isin(qs_to_remove)]

Pour convertir un DataFrame pandas vers un array numpy, utilisez `to_numpy()` qui est préférable à l'ancienne méthode `.values`.

In [ ]:
# Conversion vers numpy avec to_numpy() (recommandé)
questions_array = questions.to_numpy()
print("Type de l'array numpy:", questions_array.dtype)
print("Forme de l'array:", questions_array.shape)
print("Premières valeurs:", questions_array[:5, :5])

Avec des opérateurs de comparaison, on obtient des séries de booléens:

In [ ]:
df_with_total["total"] > 22

On peut utiliser ces séries de booléens pour sélectionner des lignes d'un `DataFrame`, avec le «boolean indexing». C'est très pratique pour filter les données:

In [ ]:
df[df_with_total["total"] > 22]

# Enchaînement d'opérations pandas

Souvent, on enchaîne les opérations sur les `DataFrame`s pour arriver au résultat souhaité. Pour améliorer la lisibilité, on peut utiliser des parenthèses pour enchaîner les opérations sur plusieurs lignes.

In [ ]:
# Exemple d'enchaînement multilignes avec parenthèses
correlation_with_total = (
    df_with_total[[*qs, "total"]]
    .corr()["total"]
    .drop("total")  # Enlever l'auto-corrélation
    .sort_values(ascending=False)
)

print("Questions les plus corrélées avec le total:")
correlation_with_total.head(10)

Il y a beaucoup d'autres fonctions… <https://pandas.pydata.org/docs/user_guide/index.html>